# Cross-asset exploration

This offline example uses normalized security, pair, index, commodity, economic, quote, and top-of-book data. It makes alignment and transformation choices explicit.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import pandas as pd

from persistra.analysis import (
    absolute_spread,
    growth_rate,
    midprice,
    simple_returns,
    summary_statistics,
)
from persistra.data import DuckDBStore, align, pivot_bars, pivot_series, synthetic
from persistra.model import InstrumentKind, SeriesKind
from persistra.viz import (
    plot_correlation,
    plot_rebased,
    plot_scalar_series,
    plot_spread_history,
)

In [ ]:
equity = synthetic.bars("EQUITY", periods=36, seed=1)
fx = synthetic.bars("EUR/USD", periods=36, seed=2, kind=InstrumentKind.FIAT_PAIR)
crypto = synthetic.bars("BTC/USD", periods=36, seed=3, kind=InstrumentKind.CRYPTO_PAIR)
index = synthetic.bars("INDEX", periods=36, seed=4, kind=InstrumentKind.INDEX)
prices = pivot_bars([equity, fx, crypto, index], field="close")
returns = simple_returns(prices)
summary_statistics(returns).round(4)

In [ ]:
commodity = synthetic.series("WTI", periods=24, kind=SeriesKind.COMMODITY)
economic = synthetic.series("CPI", periods=24, kind=SeriesKind.ECONOMIC)
levels = pivot_series([commodity, economic])
levels.index = pd.to_datetime(levels.index)
explicit_union = align({"prices": prices, "series": levels}, how="union")
series_growth = growth_rate(levels, lag=1)
assert explicit_union["prices"].isna().any().any()
series_growth.tail()

In [ ]:
quotes = synthetic.quotes(("EQUITY", "FUND"))
book = synthetic.top_of_book(("EQUITY", "FUND"))
book_later = book.frame.copy()
book_later["observed_at"] += pd.Timedelta(minutes=5)
book_later["bid_price"] -= 0.05
book_later["ask_price"] += 0.05
book_history = pd.concat([book.frame, book_later], ignore_index=True)
book_levels = midprice(book)
book_spreads = absolute_spread(book)
temporary_directory = TemporaryDirectory()
store_path = Path(temporary_directory.name) / "research.duckdb"
with DuckDBStore.create(store_path) as store:
    for result in (equity, fx, crypto, index, commodity, economic, quotes, book):
        store.save(result)
    restored = store.load_bars(equity.instrument.instrument_id)
assert restored is not None and restored.frame.equals(equity.frame)

In [ ]:
plot_rebased(prices)
plot_correlation(returns)
plot_scalar_series(economic)
plot_spread_history(book_history[book_history["provider_symbol"] == "EQUITY"])
assert book_levels["midprice"].notna().all()
assert book_spreads["absolute_spread"].notna().all()
plt.close("all")
temporary_directory.cleanup()